# LeetCode 1253 — Reconstruct a 2-Row Binary Matrix bằng Python

Link đề: https://leetcode.com/problems/reconstruct-a-2-row-binary-matrix/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Matrix

Skeleton LeetCode:

```python
class Solution:
    def reconstructMatrix(self, upper: int, lower: int, colsum: List[int]) -> List[List[int]]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng**; nếu không có lời giải thì trả về **mảng 2 chiều rỗng** `[]`. Khi kiểm thử ta dùng **hàm kiểm tra hợp lệ** thay vì so khớp một ma trận cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dựng ma trận nhị phân 2 hàng × `n` cột biết tổng hàng trên `upper`, tổng hàng dưới `lower`, tổng từng cột `colsum[i] ∈ {0, 1, 2}`.
2. Nhận ra cột `0` và cột `2` là **bị ép buộc** (forced); chỉ cột `1` mới có lựa chọn (đặt 1 ở trên hay ở dưới).
3. Rút ra **điều kiện có nghiệm**: `upper + lower == sum(colsum)` và `số cột 2 ≤ min(upper, lower)`.
4. Viết lời giải tham lam `O(n)` theo nhiều kiểu (đếm trước rồi phân bổ; tham lam "hàng nào còn nhiều hơn thì nhận"), và vét cạn để đối chiếu.
5. Tránh các lỗi: tạo ma trận bằng `[[0] * n] * 2` (hai hàng dùng chung), tham lam thiếu ưu tiên cột 2, quên kiểm tra tổng ở cuối, trả về `[[], []]` thay vì `[]`.

## 2. Đề bài & ví dụ

Cho thông tin về một ma trận có `n` cột và **2 hàng**:

- Ma trận **nhị phân**: mỗi phần tử là `0` hoặc `1`.
- Tổng các phần tử của hàng 0 (hàng trên) là `upper`.
- Tổng các phần tử của hàng 1 (hàng dưới) là `lower`.
- Tổng các phần tử của cột `i` là `colsum[i]` (`colsum` dài `n`).

Hãy dựng lại ma trận và trả về dưới dạng mảng 2 chiều. Nếu có nhiều đáp án, trả về đáp án bất kỳ. Nếu không có đáp án, trả về mảng 2 chiều **rỗng**.

**Ví dụ 1:** `upper = 2, lower = 1, colsum = [1,1,1]` → `[[1,1,0],[0,0,1]]` (cũng đúng: `[[1,0,1],[0,1,0]]`, `[[0,1,1],[1,0,0]]`).

**Ví dụ 2:** `upper = 2, lower = 3, colsum = [2,2,1,1]` → `[]`.

**Ví dụ 3:** `upper = 5, lower = 5, colsum = [2,1,2,0,1,0,1,2,0,1]` → `[[1,1,1,0,1,0,0,1,0,0],[1,0,1,0,0,0,1,1,0,1]]`.

**Ràng buộc:** `1 <= colsum.length <= 10⁵`; `0 <= upper, lower <= colsum.length`; `0 <= colsum[i] <= 2`.

In [1]:
import itertools
import random
import time
from typing import List

examples = [
    (2, 1, [1, 1, 1], [[1, 1, 0], [0, 0, 1]]),
    (2, 3, [2, 2, 1, 1], []),
    (5, 5, [2, 1, 2, 0, 1, 0, 1, 2, 0, 1], [[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]]),
]
for u, l, cs, sample in examples:
    print(f"upper={u}, lower={l}, colsum={cs} -> {sample}")

upper=2, lower=1, colsum=[1, 1, 1] -> [[1, 1, 0], [0, 0, 1]]
upper=2, lower=3, colsum=[2, 2, 1, 1] -> []
upper=5, lower=5, colsum=[2, 1, 2, 0, 1, 0, 1, 2, 0, 1] -> [[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]]


## 3. Kiến thức nền

### 3.1 Tạo ma trận 2 hàng đúng cách

`[[0] * n for _ in range(2)]` tạo **hai list độc lập**. Còn `[[0] * n] * 2` tạo **hai tham chiếu tới cùng một list** — sửa hàng này thì hàng kia đổi theo (bẫy kinh điển).

In [2]:
n = 3
good = [[0] * n for _ in range(2)]
bad = [[0] * n] * 2
good[0][1] = 1
bad[0][1] = 1
print("good:", good)
print("bad :", bad, "| cùng một object?", bad[0] is bad[1])

good: [[0, 1, 0], [0, 0, 0]]
bad : [[0, 1, 0], [0, 1, 0]] | cùng một object? True


### 3.2 Tổng hàng, tổng cột với `sum` và `zip`

`sum(row)` cho tổng hàng; `zip(*M)` "xoay" ma trận để duyệt theo cột, hoặc với 2 hàng chỉ cần `M[0][i] + M[1][i]`.

In [3]:
M = [[1, 1, 0], [0, 0, 1]]
print("tổng hàng:", [sum(r) for r in M])
print("tổng cột :", [a + b for a, b in zip(*M)])

tổng hàng: [2, 1]
tổng cột : [1, 1, 1]


### 3.3 Đếm phần tử với `list.count`

`colsum.count(2)` cho số cột tổng 2 — chi phí `O(n)`.

In [4]:
cs = [2, 1, 2, 0, 1, 0, 1, 2, 0, 1]
print("số cột 2:", cs.count(2), "| số cột 1:", cs.count(1), "| tổng:", sum(cs))

số cột 2: 3 | số cột 1: 4 | tổng: 10


### 3.4 Validator cho bài nhiều đáp án

`is_valid_matrix(upper, lower, colsum, M)` kiểm tra: đúng 2 hàng, mỗi hàng dài `n`, mọi phần tử là 0/1, tổng hàng và tổng cột khớp. Với trường hợp `M == []`, ta cần biết bài **thật sự vô nghiệm** — dùng hàm `feasible` (sẽ chứng minh ở mục 6, và đối chiếu bằng vét cạn ở mục 12).

In [5]:
def is_valid_matrix(upper, lower, colsum, M):
    n = len(colsum)
    if len(M) != 2 or any(len(r) != n for r in M):
        return False
    if any(x not in (0, 1) for r in M for x in r):
        return False
    if sum(M[0]) != upper or sum(M[1]) != lower:
        return False
    return all(M[0][i] + M[1][i] == colsum[i] for i in range(n))


def feasible(upper, lower, colsum):
    twos = colsum.count(2)
    return upper + lower == sum(colsum) and twos <= min(upper, lower)


def is_correct(upper, lower, colsum, res):
    if res == []:
        return not feasible(upper, lower, colsum)
    return is_valid_matrix(upper, lower, colsum, res)


for u, l, cs, sample in examples:
    print(u, l, cs, "-> feasible:", feasible(u, l, cs), "| mẫu đúng?", is_correct(u, l, cs, sample))

2 1 [1, 1, 1] -> feasible: True | mẫu đúng? True
2 3 [2, 2, 1, 1] -> feasible: False | mẫu đúng? True
5 5 [2, 1, 2, 0, 1, 0, 1, 2, 0, 1] -> feasible: True | mẫu đúng? True


## 4. Ý tưởng nền tảng / trực giác

Nhìn từng loại cột:

- `colsum[i] = 0` → bắt buộc `(0, 0)`.
- `colsum[i] = 2` → bắt buộc `(1, 1)` — **cả hai hàng** đều mất 1 đơn vị "ngân sách".
- `colsum[i] = 1` → `(1, 0)` hoặc `(0, 1)` — đây là chỗ duy nhất có lựa chọn.

Tính tay ví dụ 3: `colsum = [2,1,2,0,1,0,1,2,0,1]`, `upper = lower = 5`.

- Có 3 cột `2` → mỗi hàng đã dùng 3; còn `upper' = 2`, `lower' = 2`.
- Có 4 cột `1` → cần chia 2 cho trên, 2 cho dưới: `2 + 2 = 4` ✓.

Ví dụ 2: `colsum = [2,2,1,1]` có tổng 6 nhưng `upper + lower = 5` → vô nghiệm.

In [6]:
u, l, cs = 5, 5, [2, 1, 2, 0, 1, 0, 1, 2, 0, 1]
twos, ones = cs.count(2), cs.count(1)
print(f"twos={twos}, ones={ones}, upper'={u - twos}, lower'={l - twos}, upper'+lower'={u + l - 2 * twos} so với ones={ones}")

twos=3, ones=4, upper'=2, lower'=2, upper'+lower'=4 so với ones=4


## 5. Cách 1 — Brute Force / baseline

Cột 0 và 2 đã cố định; với `m` cột tổng 1, thử **mọi** cách gán `2ᵐ` (bit `1` = đặt ở hàng trên). Trả về cách đầu tiên khớp `upper`/`lower`.

In [7]:
class SolutionBrute:
    def reconstructMatrix(self, upper, lower, colsum):
        n = len(colsum)
        ones_idx = [i for i in range(n) if colsum[i] == 1]
        for mask in range(1 << len(ones_idx)):
            top = [1 if c == 2 else 0 for c in colsum]
            bot = top[:]
            for b, i in enumerate(ones_idx):
                if mask >> b & 1:
                    top[i] = 1
                else:
                    bot[i] = 1
            if sum(top) == upper and sum(bot) == lower:
                return [top, bot]
        return []


for u, l, cs, _ in examples:
    res = SolutionBrute().reconstructMatrix(u, l, cs)
    print(res, is_correct(u, l, cs, res))

[[1, 1, 0], [0, 0, 1]] True
[] True
[[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]] True


**Độ phức tạp:** `O(2ᵐ · n)` — với `n = 10⁵` hoàn toàn bất khả thi. **Nút thắt:** các cột `1` **hoán đổi được cho nhau** — điều duy nhất quan trọng là **bao nhiêu** cột `1` đặt ở hàng trên, không phải **cột nào**. Vậy chỉ có `m + 1` "lựa chọn thật", thậm chí chỉ một giá trị đúng.

## 6. Bước đột phá

**Insight:** gọi `t` = số cột `2`, `m` = số cột `1`. Hàng trên phải nhận thêm `upper − t` dấu `1` từ các cột `1`, hàng dưới `lower − t`. Có nghiệm ⇔

1. `upper − t ≥ 0` và `lower − t ≥ 0` (đủ ngân sách cho các cột `2`), và
2. `(upper − t) + (lower − t) = m` (các cột `1` chia vừa khít) ⇔ `upper + lower = 2t + m = sum(colsum)`.

Khi đó **bất kỳ** cách chọn `upper − t` cột `1` cho hàng trên đều đúng.

**Quy tắc tham lam:**

- Nếu cột là `2` → `(1, 1)`, `upper -= 1`, `lower -= 1`.
- Nếu cột là `1` → đặt vào hàng **còn nhiều ngân sách hơn** (hoặc: đặt hàng trên khi hàng trên còn ngân sách **sau khi đã trừ trước các cột 2**).
- Nếu ngân sách âm → vô nghiệm; cuối cùng ngân sách phải về đúng `0`.

**Invariant (cho cách "đếm trước"):** sau khi trừ trước toàn bộ cột `2`, `need_up` = số cột `1` còn phải giao cho hàng trên; mỗi cột `1` giao cho trên khi `need_up > 0`, ngược lại giao cho dưới.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm trước, kiểm tra điều kiện, rồi phân bổ

In [8]:
class SolutionCountFirst:
    def reconstructMatrix(self, upper, lower, colsum):
        t = colsum.count(2)
        m = colsum.count(1)
        if upper < t or lower < t or upper + lower != 2 * t + m:
            return []
        need_up = upper - t                 # số cột 1 giao cho hàng trên
        n = len(colsum)
        top, bot = [0] * n, [0] * n
        for i, c in enumerate(colsum):
            if c == 2:
                top[i] = bot[i] = 1
            elif c == 1:
                if need_up > 0:
                    top[i] = 1
                    need_up -= 1
                else:
                    bot[i] = 1
        return [top, bot]


for u, l, cs, _ in examples:
    res = SolutionCountFirst().reconstructMatrix(u, l, cs)
    print(res, is_correct(u, l, cs, res))

[[1, 1, 0], [0, 0, 1]] True
[] True
[[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]] True


**Độ phức tạp:** `O(n)` thời gian (vài lượt duyệt), `O(n)` cho kết quả.

### 7.2 Cách 3 — Tham lam một lượt: cột 1 cho hàng còn nhiều hơn

Không đếm trước. Mỗi cột `2` trừ cả hai; mỗi cột `1` giao cho hàng có ngân sách còn lại **lớn hơn**. Nếu bất kỳ lúc nào ngân sách âm → `[]`. Cuối cùng cả hai phải bằng `0`.

In [9]:
class SolutionGreedyBalance:
    def reconstructMatrix(self, upper, lower, colsum):
        n = len(colsum)
        top, bot = [0] * n, [0] * n
        for i, c in enumerate(colsum):
            if c == 2:
                top[i] = bot[i] = 1
                upper -= 1
                lower -= 1
            elif c == 1:
                if upper >= lower:
                    top[i] = 1
                    upper -= 1
                else:
                    bot[i] = 1
                    lower -= 1
            if upper < 0 or lower < 0:
                return []
        return [top, bot] if upper == 0 and lower == 0 else []


for u, l, cs, _ in examples:
    res = SolutionGreedyBalance().reconstructMatrix(u, l, cs)
    print(res, is_correct(u, l, cs, res))

[[1, 1, 0], [0, 0, 1]] True
[] True
[[1, 1, 1, 0, 0, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0, 0, 1, 0, 1]] True


Vì sao "hàng còn nhiều hơn" an toàn dù chưa thấy các cột `2` phía sau? Cách chọn này giữ hiệu `|upper − lower|` nhỏ nhất có thể, nên `min(upper, lower)` còn lại lớn nhất có thể cho các cột `2` sắp tới. (Chứng minh ở mục 10; kiểm chứng bằng vét cạn ở mục 12.)

**Độ phức tạp:** `O(n)` một lượt, dừng sớm khi phát hiện vô nghiệm.

### 7.3 Cách 4 — Hai lượt: xử lý cột 2 trước, rồi cột 1

Một cách viết "an toàn" khác: lượt 1 chỉ đặt các cột `2` và trừ ngân sách; lượt 2 đặt các cột `1` (ưu tiên hàng trên cho tới khi hết ngân sách trên). Không cần lập luận tinh tế như Cách 3.

In [10]:
class SolutionTwoPass:
    def reconstructMatrix(self, upper, lower, colsum):
        n = len(colsum)
        M = [[0] * n for _ in range(2)]
        for i, c in enumerate(colsum):
            if c == 2:
                M[0][i] = M[1][i] = 1
                upper -= 1
                lower -= 1
        if upper < 0 or lower < 0:
            return []
        for i, c in enumerate(colsum):
            if c == 1:
                if upper > 0:
                    M[0][i] = 1
                    upper -= 1
                elif lower > 0:
                    M[1][i] = 1
                    lower -= 1
                else:
                    return []
        return M if upper == 0 and lower == 0 else []


for u, l, cs, _ in examples:
    res = SolutionTwoPass().reconstructMatrix(u, l, cs)
    print(res, is_correct(u, l, cs, res))

[[1, 1, 0], [0, 0, 1]] True
[] True
[[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]] True


### 7.4 Cách 5 — Dựng bằng công thức "k cột 1 đầu tiên lên trên"

Sau khi kiểm tra điều kiện, hàng trên = cột `2` **hoặc** (cột `1` nằm trong `need_up` cột `1` đầu tiên); hàng dưới = `colsum[i] − top[i]`. Viết gọn bằng comprehension và một bộ đếm thứ tự cột `1`.

In [11]:
class SolutionFormula:
    def reconstructMatrix(self, upper, lower, colsum):
        t, total = colsum.count(2), sum(colsum)
        if total != upper + lower or t > min(upper, lower):
            return []
        need_up = upper - t
        top, seen_one = [], 0
        for c in colsum:
            if c == 1:
                seen_one += 1
                top.append(1 if seen_one <= need_up else 0)
            else:
                top.append(c // 2)          # 0 -> 0, 2 -> 1
        bot = [c - x for c, x in zip(colsum, top)]
        return [top, bot]


for u, l, cs, _ in examples:
    res = SolutionFormula().reconstructMatrix(u, l, cs)
    print(res, is_correct(u, l, cs, res))

[[1, 1, 0], [0, 0, 1]] True
[] True
[[1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1]] True


## 8. TRACE từng bước

Trace Cách 3 (tham lam cân bằng) trên ví dụ 3: `upper = lower = 5`, `colsum = [2,1,2,0,1,0,1,2,0,1]`.

| i | colsum | so sánh | đặt | upper | lower |
|---|---|---|---|---|---|
| 0 | 2 | — | (1,1) | 4 | 4 |
| 1 | 1 | 4 ≥ 4 | trên | 3 | 4 |
| 2 | 2 | — | (1,1) | 2 | 3 |
| 3 | 0 | — | (0,0) | 2 | 3 |
| 4 | 1 | 2 < 3 | dưới | 2 | 2 |
| 5 | 0 | — | (0,0) | 2 | 2 |
| 6 | 1 | 2 ≥ 2 | trên | 1 | 2 |
| 7 | 2 | — | (1,1) | 0 | 1 |
| 8 | 0 | — | (0,0) | 0 | 1 |
| 9 | 1 | 0 < 1 | dưới | 0 | 0 |

Kết thúc `upper = lower = 0` → hợp lệ.

In [12]:
def trace_greedy(upper, lower, colsum):
    n = len(colsum)
    top, bot = [0] * n, [0] * n
    for i, c in enumerate(colsum):
        if c == 2:
            top[i] = bot[i] = 1
            upper -= 1
            lower -= 1
            act = "(1,1)"
        elif c == 1:
            if upper >= lower:
                top[i] = 1
                upper -= 1
                act = "trên"
            else:
                bot[i] = 1
                lower -= 1
                act = "dưới"
        else:
            act = "(0,0)"
        print(f"i={i}: colsum={c} -> {act:5} | upper={upper}, lower={lower}")
        if upper < 0 or lower < 0:
            print("ngân sách âm -> []")
            return []
    ok = upper == 0 and lower == 0
    print("kết quả:", [top, bot] if ok else "[] (ngân sách chưa về 0)")
    return [top, bot] if ok else []


trace_greedy(5, 5, [2, 1, 2, 0, 1, 0, 1, 2, 0, 1])

i=0: colsum=2 -> (1,1) | upper=4, lower=4
i=1: colsum=1 -> trên  | upper=3, lower=4
i=2: colsum=2 -> (1,1) | upper=2, lower=3
i=3: colsum=0 -> (0,0) | upper=2, lower=3
i=4: colsum=1 -> dưới  | upper=2, lower=2
i=5: colsum=0 -> (0,0) | upper=2, lower=2
i=6: colsum=1 -> trên  | upper=1, lower=2
i=7: colsum=2 -> (1,1) | upper=0, lower=1
i=8: colsum=0 -> (0,0) | upper=0, lower=1
i=9: colsum=1 -> dưới  | upper=0, lower=0
kết quả: [[1, 1, 1, 0, 0, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0, 0, 1, 0, 1]]


[[1, 1, 1, 0, 0, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0, 0, 1, 0, 1]]

Trace ví dụ 2 (vô nghiệm):

In [13]:
trace_greedy(2, 3, [2, 2, 1, 1])

i=0: colsum=2 -> (1,1) | upper=1, lower=2
i=1: colsum=2 -> (1,1) | upper=0, lower=1
i=2: colsum=1 -> dưới  | upper=0, lower=0
i=3: colsum=1 -> trên  | upper=-1, lower=0
ngân sách âm -> []


[]

## 9. Code nộp LeetCode

In [14]:
from typing import List


class Solution:
    def reconstructMatrix(self, upper: int, lower: int, colsum: List[int]) -> List[List[int]]:
        t = colsum.count(2)
        if upper + lower != sum(colsum) or t > min(upper, lower):
            return []
        need_up = upper - t
        n = len(colsum)
        top, bot = [0] * n, [0] * n
        for i, c in enumerate(colsum):
            if c == 2:
                top[i] = bot[i] = 1
            elif c == 1:
                if need_up > 0:
                    top[i] = 1
                    need_up -= 1
                else:
                    bot[i] = 1
        return [top, bot]


for u, l, cs, _ in examples:
    assert is_correct(u, l, cs, Solution().reconstructMatrix(u, l, cs))
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

**Điều kiện cần.** Nếu có ma trận hợp lệ: tổng mọi phần tử tính theo hàng là `upper + lower`, tính theo cột là `sum(colsum)` → bằng nhau. Mỗi cột `2` đóng góp 1 cho **mỗi** hàng → `upper ≥ t` và `lower ≥ t`.

**Điều kiện đủ (và Cách 2/5 đúng).** Khi hai điều kiện thoả: `need_up = upper − t ≥ 0`, `need_down = lower − t ≥ 0`, và `need_up + need_down = upper + lower − 2t = sum(colsum) − 2t = m`. Giao đúng `need_up` cột `1` cho hàng trên, `m − need_up = need_down` cột cho hàng dưới: tổng hàng trên `t + need_up = upper`, hàng dưới `t + need_down = lower`, tổng cột đúng theo cách dựng. ∎

**Cách 3 (tham lam cân bằng) đúng.** Giả sử bài có nghiệm. Ta chứng minh ngân sách không bao giờ âm. Gọi `U`, `L` là ngân sách hiện tại; `T` = số cột `2` còn lại phía sau, `M` = số cột `1` còn lại (gồm cột hiện tại). Bất biến: `U + L = 2T + M` (do tổng khớp) và `min(U, L) ≥ T`. Khi gặp cột `1` và giao cho hàng lớn hơn, giả sử `U ≥ L`: nếu `U − 1 < T` thì `U ≤ T`, kết hợp `L ≤ U ≤ T` và `U + L = 2T + M ≥ 2T + 1` → mâu thuẫn. Vậy `min` vẫn `≥ T`. Cột `2` giảm cả `U`, `L` và `T` đi 1 → giữ bất biến. Cuối cùng `T = M = 0` → `U + L = 0` với `U, L ≥ 0` → cả hai bằng 0. Nếu bài vô nghiệm, thuật toán hoặc gặp ngân sách âm hoặc kết thúc với ngân sách khác 0 → trả `[]` đúng.

## 11. Time / Space Complexity

Đặt `n = len(colsum)`, `m` = số cột `1`.

| Cách | Time | Space (ngoài kết quả `2n`) |
|---|---|---|
| 1. Brute | `O(2ᵐ · n)` | `O(n)` |
| 2. Đếm trước rồi phân bổ | `O(n)` | `O(1)` |
| 3. Tham lam cân bằng 1 lượt | `O(n)` | `O(1)` |
| 4. Hai lượt (cột 2 trước) | `O(n)` | `O(1)` |
| 5. Công thức | `O(n)` | `O(1)` |

Kết quả bản thân chiếm `2n` ô.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [15]:
ALL = [SolutionCountFirst, SolutionGreedyBalance, SolutionTwoPass, SolutionFormula, Solution]


def check(u, l, cs, with_brute=True):
    outs = [cls().reconstructMatrix(u, l, cs) for cls in ALL]
    for cls, o in zip(ALL, outs):
        assert is_correct(u, l, cs, o), (cls.__name__, u, l, cs, o)
    if with_brute:
        b = SolutionBrute().reconstructMatrix(u, l, cs)
        assert (b == []) == (outs[0] == []), (u, l, cs)   # brute xác nhận feasible
        assert is_correct(u, l, cs, b)
    return outs[0]


for u, l, cs, _ in examples:
    check(u, l, cs)
print(check(0, 0, [0, 0]))                 # toàn 0
print(check(1, 0, [1]))
print(check(0, 1, [1]))
print(check(1, 1, [2]))
print(check(1, 1, [1, 1]))
print(check(2, 2, [1, 1, 2]))              # cột 2 ở cuối
print(check(0, 2, [2]))                    # thiếu ngân sách hàng trên -> []
print(check(3, 0, [1, 1, 1]))
print(check(1, 1, [0, 0]))                 # tổng không khớp -> []

[[0, 0], [0, 0]]
[[1], [0]]
[[0], [1]]
[[1], [1]]
[[1, 0], [0, 1]]
[[1, 0, 1], [0, 1, 1]]
[]
[[1, 1, 1], [0, 0, 0]]
[]


### 12.2 Ngẫu nhiên có seed: đối chiếu với vét cạn

Vét cạn cũng giúp kiểm chứng hàm `feasible` (điều kiện ở mục 6) — vì `is_correct` coi `[]` là đúng chỉ khi `feasible` sai, còn brute thì không dùng `feasible`.

In [16]:
rng = random.Random(1253)
n_feasible = 0
for _ in range(1500):
    n = rng.randint(1, 10)
    cs = [rng.randint(0, 2) for _ in range(n)]
    if rng.random() < 0.6:                 # sinh upper/lower có nghiệm với xác suất cao
        t, m = cs.count(2), cs.count(1)
        k = rng.randint(0, m)
        u, l = t + k, t + m - k
        u += rng.choice([0, 0, 0, 1, -1])
        u = max(0, min(n, u))
    else:
        u, l = rng.randint(0, n), rng.randint(0, n)
    b = SolutionBrute().reconstructMatrix(u, l, cs)
    assert (b != []) == feasible(u, l, cs)
    n_feasible += b != []
    check(u, l, cs)
print("1500 test ngẫu nhiên OK, số test có nghiệm:", n_feasible)

1500 test ngẫu nhiên OK, số test có nghiệm: 655


### 12.3 Test giới hạn `n = 10⁵`

In [17]:
rng = random.Random(5)
cs = [rng.randint(0, 2) for _ in range(10 ** 5)]
t, m = cs.count(2), cs.count(1)
cases = [(t + m // 2, t + m - m // 2, cs), (t + m, t, cs), (t, t + m, cs), (t - 1, t + m + 1, cs)]
for u, l, c in cases:
    tm = time.perf_counter()
    res = check(u, l, c, with_brute=False)
    print(f"upper={u}, lower={l}: {'có nghiệm' if res else '[]'} ({(time.perf_counter() - tm) * 1000:.0f} ms)")

upper=50004, lower=50004: có nghiệm (111 ms)


upper=66762, lower=33246: có nghiệm (111 ms)


upper=33246, lower=66762: có nghiệm (105 ms)
upper=33245, lower=66763: [] (26 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tạo ma trận bằng `[[0] * n] * 2`

In [18]:
def wrong_alias(upper, lower, colsum):
    t = colsum.count(2)
    if upper + lower != sum(colsum) or t > min(upper, lower):
        return []
    need_up = upper - t
    M = [[0] * len(colsum)] * 2           # SAI: hai hàng là cùng một list
    for i, c in enumerate(colsum):
        if c == 2:
            M[0][i] = M[1][i] = 1
        elif c == 1:
            if need_up > 0:
                M[0][i] = 1
                need_up -= 1
            else:
                M[1][i] = 1
    return M


res = wrong_alias(2, 1, [1, 1, 1])
print("sai :", res, is_valid_matrix(2, 1, [1, 1, 1], res))
res = Solution().reconstructMatrix(2, 1, [1, 1, 1])
print("đúng:", res, is_valid_matrix(2, 1, [1, 1, 1], res))

sai : [[1, 1, 1], [1, 1, 1]] False
đúng: [[1, 1, 0], [0, 0, 1]] True


**Vì sao sai:** `[[0] * n] * 2` nhân bản **tham chiếu**, nên gán `M[1][i]` cũng sửa `M[0][i]` → hai hàng giống hệt nhau. **Sửa:** `[[0] * n for _ in range(2)]` hoặc hai list riêng `top`, `bot`.

### Lỗi 2 — Tham lam "cột 1 luôn ưu tiên hàng trên" ngay trong một lượt

In [19]:
def wrong_greedy_upper_first(upper, lower, colsum):
    n = len(colsum)
    top, bot = [0] * n, [0] * n
    for i, c in enumerate(colsum):
        if c == 2:
            top[i] = bot[i] = 1
            upper -= 1
            lower -= 1
        elif c == 1:
            if upper > 0:                   # SAI: tiêu ngân sách trên trước khi gặp cột 2
                top[i] = 1
                upper -= 1
            else:
                bot[i] = 1
                lower -= 1
        if upper < 0 or lower < 0:
            return []
    return [top, bot] if upper == lower == 0 else []


print("sai :", wrong_greedy_upper_first(2, 2, [1, 1, 2]))
print("đúng:", Solution().reconstructMatrix(2, 2, [1, 1, 2]))

sai : []
đúng: [[1, 0, 1], [0, 1, 1]]


**Vì sao sai:** hai cột `1` đầu tiêu hết ngân sách hàng trên; đến cột `2` thì `upper = −1` → trả `[]` dù có nghiệm `[[1,0,1],[0,1,1]]`. **Sửa:** trừ trước ngân sách cho **mọi** cột `2` (Cách 2/4/5), hoặc giao cột `1` cho hàng **còn nhiều hơn** (Cách 3).

### Lỗi 3 — Quên kiểm tra ngân sách về 0 ở cuối

In [20]:
def wrong_no_final_check(upper, lower, colsum):
    n = len(colsum)
    top, bot = [0] * n, [0] * n
    for i, c in enumerate(colsum):
        if c == 2:
            top[i] = bot[i] = 1
            upper -= 1
            lower -= 1
        elif c == 1:
            if upper >= lower:
                top[i] = 1
                upper -= 1
            else:
                bot[i] = 1
                lower -= 1
        if upper < 0 or lower < 0:
            return []
    return [top, bot]                       # SAI: không kiểm tra upper == lower == 0


res = wrong_no_final_check(1, 1, [0, 0])
print("sai :", res, "| hợp lệ?", is_correct(1, 1, [0, 0], res))
print("đúng:", Solution().reconstructMatrix(1, 1, [0, 0]))

sai : [[0, 0], [0, 0]] | hợp lệ? False
đúng: []


**Vì sao sai:** ngân sách còn dư (`upper = lower = 1`) nghĩa là tổng hàng không khớp; ma trận toàn 0 không có hàng nào tổng bằng 1. **Sửa:** cuối vòng lặp kiểm tra `upper == 0 and lower == 0`, hoặc kiểm tra `upper + lower == sum(colsum)` ngay từ đầu.

### Lỗi 4 — Trả về `[[], []]` thay vì `[]` khi vô nghiệm

In [21]:
def wrong_empty(upper, lower, colsum):
    res = Solution().reconstructMatrix(upper, lower, colsum)
    return res if res else [[], []]         # SAI: đề yêu cầu mảng 2 chiều rỗng []


print("sai :", wrong_empty(2, 3, [2, 2, 1, 1]), "| đúng:", Solution().reconstructMatrix(2, 3, [2, 2, 1, 1]))
print("[[], []] == [] ?", [[], []] == [])

sai : [[], []] | đúng: []
[[], []] == [] ? False


**Vì sao sai:** trình chấm so sánh với `[]`; `[[], []]` là list có 2 phần tử, khác `[]`. **Sửa:** `return []`.

## 14. Mẹo & nhận dạng pattern

- Bài "dựng lại ma trận/dãy từ tổng hàng/cột" → tách phần **bị ép buộc** (forced) khỏi phần **có lựa chọn**; phần có lựa chọn thường hoán đổi được → chỉ cần **đếm**.
- Kiểm tra **điều kiện cần** đơn giản trước (tổng khớp, ngân sách đủ) — thường chúng cũng là **điều kiện đủ**.
- Tham lam một lượt: ưu tiên phần tử "khó" (cột 2) hoặc giữ trạng thái **cân bằng** để không tự chặn đường.
- Bài nhiều đáp án → viết validator; dùng vét cạn để kiểm chứng điều kiện có nghiệm.
- Cẩn thận `[[0] * n] * k` (aliasing).
- Bài cùng họ: 1605 Find Valid Matrix Given Row and Column Sums, 2375 Construct Smallest Number From DI String, Gale–Ryser (bài toán ma trận 0/1 với tổng hàng/cột tổng quát).

## 15. Biến thể

1. **`k` hàng thay vì 2** (tổng hàng `r[0..k−1]`, tổng cột `c[j] ∈ [0, k]`) → định lý Gale–Ryser; tham lam: mỗi cột đặt `1` vào `c[j]` hàng còn nhiều ngân sách nhất (dùng heap).
2. **Đếm số ma trận hợp lệ** → `C(m, upper − t)` khi có nghiệm.
3. **Ma trận nhỏ nhất theo thứ tự từ điển** (hàng trên lớn nhất) → giao các cột `1` **đầu tiên** cho hàng trên (đúng như Cách 2).
4. **1605 — số nguyên không âm tuỳ ý**: tham lam `M[i][j] = min(row[i], col[j])`.
5. **Có ô bị khoá bằng 0** → thành bài luồng cực đại (max flow) / ghép cặp.

Demo biến thể 2 và 1:

In [22]:
from math import comb
import heapq


def count_matrices(upper, lower, colsum):
    if not feasible(upper, lower, colsum):
        return 0
    return comb(colsum.count(1), upper - colsum.count(2))


def brute_count(upper, lower, colsum):
    idx = [i for i, c in enumerate(colsum) if c == 1]
    t = colsum.count(2)
    return sum(1 for mask in range(1 << len(idx))
               if t + bin(mask).count("1") == upper and t + len(idx) - bin(mask).count("1") == lower)


def reconstruct_k_rows(rows, cols):
    k, n = len(rows), len(cols)
    M = [[0] * n for _ in range(k)]
    left = rows[:]
    for j in sorted(range(n), key=lambda j: -cols[j]):      # cột lớn trước
        best = heapq.nlargest(cols[j], range(k), key=lambda i: left[i])
        if len(best) < cols[j] or any(left[i] == 0 for i in best):
            return []
        for i in best:
            M[i][j] = 1
            left[i] -= 1
    return M if not any(left) else []


for cs, u, l in [([1, 1, 1], 2, 1), ([2, 1, 1, 0, 1], 3, 2)]:
    print("số ma trận:", count_matrices(u, l, cs), "| brute:", brute_count(u, l, cs))
M = reconstruct_k_rows([2, 1, 2], [3, 1, 1, 0])
print("k = 3 hàng:", M, "| tổng hàng", [sum(r) for r in M], "| tổng cột", [sum(c) for c in zip(*M)])

số ma trận: 3 | brute: 3
số ma trận: 3 | brute: 3
k = 3 hàng: [[1, 1, 0, 0], [1, 0, 0, 0], [1, 0, 1, 0]] | tổng hàng [2, 1, 2] | tổng cột [3, 1, 1, 0]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Thử mọi cách đặt cột 1 | `O(2ᵐ · n)` | `O(n)` | Kiểm thử, xác nhận điều kiện |
| 2. Đếm trước | Kiểm tra điều kiện, giao `upper − t` cột 1 lên trên | `O(n)` | `O(1)` phụ | Code nộp, dễ chứng minh |
| 3. Tham lam cân bằng | Cột 1 cho hàng còn nhiều hơn | `O(n)` 1 lượt | `O(1)` phụ | Khi dữ liệu đến dạng luồng (stream) |
| 4. Hai lượt | Cột 2 trước, cột 1 sau | `O(n)` | `O(1)` phụ | Dễ nghĩ, an toàn |
| 5. Công thức | `top = c//2` hoặc thứ tự cột 1, `bot = c − top` | `O(n)` | `O(1)` phụ | Viết gọn |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `Solution().reconstructMatrix(1, 2, [1, 1, 1])`?

**Bài 2 (tính tay):** `upper = 3, lower = 1, colsum = [2, 2, 0]` có nghiệm không? Vì sao?

**Bài 3 (điền TODO):** viết `can_reconstruct(upper, lower, colsum)` trả về `True/False` (không dựng ma trận).

**Bài 4 (tự viết lại):** viết `reconstruct_lower_first(upper, lower, colsum)` ưu tiên giao cột 1 cho **hàng dưới**.

**Bài 5 (mini challenge):** viết `reconstruct_lex_max(upper, lower, colsum)` trả về đáp án có **hàng trên lớn nhất theo thứ tự từ điển** (so sánh như list), hoặc `[]`.

In [23]:
def can_reconstruct(upper, lower, colsum):
    # TODO
    raise NotImplementedError


def reconstruct_lower_first(upper, lower, colsum):
    # TODO
    raise NotImplementedError


def reconstruct_lex_max(upper, lower, colsum):
    # TODO: gợi ý — giao các cột 1 ĐẦU TIÊN cho hàng trên
    raise NotImplementedError


# Test gợi ý:
# assert can_reconstruct(5, 5, [2, 1, 2, 0, 1, 0, 1, 2, 0, 1]) and not can_reconstruct(2, 3, [2, 2, 1, 1])
# assert is_correct(2, 1, [1, 1, 1], reconstruct_lower_first(2, 1, [1, 1, 1]))
# assert reconstruct_lex_max(2, 1, [1, 1, 1]) == [[1, 1, 0], [0, 0, 1]]

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: đếm `t`, `m`, kiểm tra hai điều kiện.
- Mức 2: Bài 1: `t = 0`, `need_up = 1` → cột 1 đầu tiên lên trên.
- Mức 3: Bài 2: tổng `4 = 3 + 1` khớp nhưng `t = 2 > lower = 1` → vô nghiệm.

**Bài 3–4:**
- Mức 1: Bài 3 chính là hàm `feasible`.
- Mức 2: Bài 4: `need_down = lower − t`, giao cột 1 cho dưới khi `need_down > 0`.
- Mức 3: kiểm tra điều kiện trước rồi mới dựng.

**Bài 5:**
- Mức 1: muốn hàng trên lớn nhất theo từ điển thì số `1` của hàng trên nên nằm **càng sớm càng tốt**.
- Mức 2: các cột 2 bắt buộc là `1` ở hàng trên; với cột 1, giao cho hàng trên khi còn ngân sách.
- Mức 3: đó chính là Cách 2 / `Solution`.

## 19. Đáp án tham khảo

**Bài 1:** `[[1, 0, 0], [0, 1, 1]]`. **Bài 2:** không — cần `lower ≥ 2` cho hai cột `2`.

In [24]:
assert Solution().reconstructMatrix(1, 2, [1, 1, 1]) == [[1, 0, 0], [0, 1, 1]]
assert Solution().reconstructMatrix(3, 1, [2, 2, 0]) == []
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
def can_reconstruct(upper, lower, colsum):
    t = colsum.count(2)
    return upper + lower == sum(colsum) and t <= min(upper, lower)


def reconstruct_lower_first(upper, lower, colsum):
    if not can_reconstruct(upper, lower, colsum):
        return []
    need_down = lower - colsum.count(2)
    top, bot = [], []
    for c in colsum:
        if c == 2:
            top.append(1)
            bot.append(1)
        elif c == 1 and need_down > 0:
            top.append(0)
            bot.append(1)
            need_down -= 1
        elif c == 1:
            top.append(1)
            bot.append(0)
        else:
            top.append(0)
            bot.append(0)
    return [top, bot]


def reconstruct_lex_max(upper, lower, colsum):
    return Solution().reconstructMatrix(upper, lower, colsum)


def brute_lex_max(upper, lower, colsum):
    idx = [i for i, c in enumerate(colsum) if c == 1]
    best = []
    for mask in range(1 << len(idx)):
        top = [1 if c == 2 else 0 for c in colsum]
        for b, i in enumerate(idx):
            if mask >> b & 1:
                top[i] = 1
        bot = [c - x for c, x in zip(colsum, top)]
        if sum(top) == upper and sum(bot) == lower and (not best or top > best[0]):
            best = [top, bot]
    return best


assert can_reconstruct(5, 5, [2, 1, 2, 0, 1, 0, 1, 2, 0, 1]) and not can_reconstruct(2, 3, [2, 2, 1, 1])
assert is_correct(2, 1, [1, 1, 1], reconstruct_lower_first(2, 1, [1, 1, 1]))
assert reconstruct_lex_max(2, 1, [1, 1, 1]) == [[1, 1, 0], [0, 0, 1]]
rng = random.Random(31)
for _ in range(400):
    n = rng.randint(1, 9)
    cs = [rng.randint(0, 2) for _ in range(n)]
    u, l = rng.randint(0, n), rng.randint(0, n)
    assert can_reconstruct(u, l, cs) == (SolutionBrute().reconstructMatrix(u, l, cs) != [])
    assert is_correct(u, l, cs, reconstruct_lower_first(u, l, cs))
    assert reconstruct_lex_max(u, l, cs) == brute_lex_max(u, l, cs)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
class Solution:
    def reconstructMatrix(self, upper: int, lower: int, colsum: List[int]) -> List[List[int]]:
        t = colsum.count(2)
        if upper + lower != sum(colsum) or t > min(upper, lower):
            return []
        need_up = upper - t
        n = len(colsum)
        top, bot = [0] * n, [0] * n
        for i, c in enumerate(colsum):
            if c == 2:
                top[i] = bot[i] = 1
            elif c == 1:
                if need_up > 0:
                    top[i] = 1; need_up -= 1
                else:
                    bot[i] = 1
        return [top, bot]
```

**Checklist trước khi submit:**

- [ ] Kiểm tra `upper + lower == sum(colsum)` và `count(2) ≤ min(upper, lower)`.
- [ ] Cột 2 → `(1,1)`, cột 0 → `(0,0)`, chỉ cột 1 mới có lựa chọn.
- [ ] Không tiêu ngân sách hàng trên cho cột 1 trước khi tính tới các cột 2.
- [ ] Tạo hai hàng **độc lập** (không `[[0]*n]*2`).
- [ ] Vô nghiệm thì trả `[]`, không phải `[[], []]`.
- [ ] Test bằng validator (nhiều đáp án đúng).